In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\ARYAN\AppData\Local\Temp\ipykernel_912\3395358772.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
folder_path = r"G:\langchain project wit RAG\data\raw"

# Get all PDF files
pdf_files = [file for file in os.listdir(folder_path) if file.endswith(".pdf")]

# Print all PDF file names
print("PDF Files:")
for pdf in pdf_files:
    print(pdf)

PDF Files:
Attention Is All You Nee.pdf
Lost in the Middle How Language Models Use Long Contexts.pdf
REACT SYNERGIZING REASONING AND ACTING IN.pdf
Retrieval-Augmented Generation for.pdf
Toolformer Language Models Can Teach Themselves to Use Tool.pdf


In [3]:
folder_path = r"G:\langchain project wit RAG\data\raw"

# Get all PDF files
pdf_files = [file for file in os.listdir(folder_path) if file.endswith(".pdf")]
all_documents = []

for pdf in pdf_files:
    file_path = os.path.join(folder_path, pdf)

    loader = PyPDFLoader(file_path)
    documents = loader.load()

    print("File Name:", pdf)
    print("Pages Loaded:", len(documents))
    print("-" * 50)

    all_documents.extend(documents)

print("Total Documents Loaded:", len(all_documents))

File Name: Attention Is All You Nee.pdf
Pages Loaded: 15
--------------------------------------------------
File Name: Lost in the Middle How Language Models Use Long Contexts.pdf
Pages Loaded: 18
--------------------------------------------------
File Name: REACT SYNERGIZING REASONING AND ACTING IN.pdf
Pages Loaded: 33
--------------------------------------------------
File Name: Retrieval-Augmented Generation for.pdf
Pages Loaded: 19
--------------------------------------------------
File Name: Toolformer Language Models Can Teach Themselves to Use Tool.pdf
Pages Loaded: 17
--------------------------------------------------
Total Documents Loaded: 102


In [4]:
folder_path = r"G:\langchain project wit RAG\data\raw"

# Validate folder
if not os.path.exists(folder_path):
    raise FileNotFoundError(f"Folder not found: {folder_path}")

# Get PDF files (case-insensitive)
pdf_files = [
    file for file in os.listdir(folder_path)
    if file.lower().endswith(".pdf")
]

# Check if folder is empty
if not pdf_files:
    raise ValueError("No PDF files found in folder.")

all_documents = []
failed_files = []

for pdf in pdf_files:
    file_path = os.path.join(folder_path, pdf)

    try:
        loader = PyPDFLoader(file_path)
        documents = loader.load()

        # Add metadata
        for doc in documents:
            doc.metadata["source_file"] = pdf
            doc.metadata["full_path"] = file_path

        all_documents.extend(documents)

        print(f"✅ Loaded: {pdf}")
        print(f"Pages: {len(documents)}")
        print("-" * 50)

    except Exception as e:
        failed_files.append(pdf)
        print(f"❌ Failed: {pdf}")
        print(f"Error: {e}")
        print("-" * 50)

print("\n========== SUMMARY ==========")
print(f"Total PDFs Found: {len(pdf_files)}")
print(f"Successfully Loaded: {len(pdf_files) - len(failed_files)}")
print(f"Failed Files: {len(failed_files)}")
print(f"Total Documents Loaded: {len(all_documents)}")

if failed_files:
    print("\nFailed PDFs:")
    for file in failed_files:
        print(file)

✅ Loaded: Attention Is All You Nee.pdf
Pages: 15
--------------------------------------------------
✅ Loaded: Lost in the Middle How Language Models Use Long Contexts.pdf
Pages: 18
--------------------------------------------------
✅ Loaded: REACT SYNERGIZING REASONING AND ACTING IN.pdf
Pages: 33
--------------------------------------------------
✅ Loaded: Retrieval-Augmented Generation for.pdf
Pages: 19
--------------------------------------------------
✅ Loaded: Toolformer Language Models Can Teach Themselves to Use Tool.pdf
Pages: 17
--------------------------------------------------

========== SUMMARY ==========
Total PDFs Found: 5
Successfully Loaded: 5
Failed Files: 0
Total Documents Loaded: 102


In [5]:
all_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'G:\\langchain project wit RAG\\data\\raw\\Attention Is All You Nee.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'Attention Is All You Nee.pdf', 'full_path': 'G:\\langchain project wit RAG\\data\\raw\\Attention Is All You Nee.pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszk

In [6]:
#from langchain.text_splitter import RecursiveCharacterTextSplitter
import uuid
import hashlib


def split_documents(
    documents,
    chunk_size=800,
    chunk_overlap=100,
    min_chunk_size=100
):
    """
    Production-grade document splitting for RAG.
    """

    # ==========================
    # Validation
    # ==========================
    if not documents:
        raise ValueError(
            "Documents list is empty."
        )

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must be smaller "
            "than chunk_size"
        )

    # ==========================
    # Token-aware splitter
    # ==========================
    try:
        text_splitter = (
            RecursiveCharacterTextSplitter
            .from_tiktoken_encoder(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
                separators=[
                    "\n# ",
                    "\n## ",
                    "\n### ",
                    "\n#### ",
                    "\n\n",
                    "\n",
                    ". ",
                    "! ",
                    "? ",
                    "; ",
                    ", ",
                    " ",
                    ""
                ]
            )
        )

    except Exception:
        print(
            "Warning: tiktoken not found. "
            "Using character splitter."
        )

        text_splitter = (
            RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap
            )
        )

    split_docs = text_splitter.split_documents(
        documents
    )

    clean_chunks = []
    seen_hashes = set()

    # ==========================
    # Cleaning + metadata
    # ==========================
    for idx, doc in enumerate(split_docs):

        content = doc.page_content.strip()

        # Remove extra whitespace
        content = " ".join(content.split())

        # Skip empty/small chunks
        if len(content) < min_chunk_size:
            continue

        # Remove duplicate chunks
        content_hash = hashlib.md5(
            content.encode()
        ).hexdigest()

        if content_hash in seen_hashes:
            continue

        seen_hashes.add(content_hash)

        doc.page_content = content

        # Better metadata
        doc.metadata.update({
            "chunk_id":
                str(uuid.uuid4()),

            "chunk_index":
                idx,

            "char_count":
                len(content),

            "word_count":
                len(content.split()),

            "estimated_tokens":
                len(content) // 4
        })

        clean_chunks.append(doc)

    # ==========================
    # Safe stats
    # ==========================
    if clean_chunks:

        chunk_lengths = [
            len(doc.page_content)
            for doc in clean_chunks
        ]

        print("=" * 60)
        print("CHUNKING SUMMARY")
        print("=" * 60)

        print(
            f"Original docs: "
            f"{len(documents)}"
        )

        print(
            f"Chunks created: "
            f"{len(clean_chunks)}"
        )

        print(
            f"Average chunk size: "
            f"{sum(chunk_lengths)//len(chunk_lengths)}"
        )

        print(
            f"Min chunk size: "
            f"{min(chunk_lengths)}"
        )

        print(
            f"Max chunk size: "
            f"{max(chunk_lengths)}"
        )

        print("=" * 60)

        print("\nExample Chunk:")
        print(
            clean_chunks[0]
            .page_content[:200]
        )

        print("\nMetadata:")
        print(
            clean_chunks[0]
            .metadata
        )

    return clean_chunks

In [7]:
chunks = split_documents(all_documents)
chunks

CHUNKING SUMMARY
Original docs: 102
Chunks created: 187
Average chunk size: 2043
Min chunk size: 479
Max chunk size: 3871

Example Chunk:
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need 

Metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'G:\\langchain project wit RAG\\data\\raw\\Attention Is All You Nee.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'Attention Is All You Nee.pdf', 'full_path': 'G:\\langchain project wit RAG\\data\\raw\\Attention Is All You Nee.pdf', 'chunk_id': 'd6ab3a69-1e67-43b4-8196-d2e10d25

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'G:\\langchain project wit RAG\\data\\raw\\Attention Is All You Nee.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'Attention Is All You Nee.pdf', 'full_path': 'G:\\langchain project wit RAG\\data\\raw\\Attention Is All You Nee.pdf', 'chunk_id': 'd6ab3a69-1e67-43b4-8196-d2e10d25db9f', 'chunk_index': 0, 'char_count': 2857, 'word_count': 396, 'estimated_tokens': 714}, page_content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brai

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity